# 2. Dataset
# =========================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

第一段：
你只需要抓住3 个核心点，剩下的细枝末节完全可以跳过:
1.transforms.Compose([...]) 就是一个按顺序执行的流水线。
它把括号里的多个操作串起来，图片会依次经过每一步处理。这里的顺序是固定的：先 ToTensor()，再 Normalize()，不能颠倒。

2.ToTensor()——格式转换 + 缩放
格式转换：把 PIL 图片或 NumPy 数组转成 PyTorch 的 Tensor 格式，形状从 (H, W, C) 变成 (C, H, W)（通道数在前）。
数值缩放：把像素值从 [0, 255] 的整数范围，自动缩放到 [0, 1] 的小数范围。

3.Normalize()——标准化（最关键）
这一步是让模型训练更稳、收敛更快的关键操作。
第二组 (0.2470, 0.2435, 0.2616)：分别是三个通道的标准差

这段代码的作用就是：把原始图片 → 转成 Tensor → 缩放到 [0,1] → 按 CIFAR-10 的统计值做标准化，让数据分布接近零均值、单位方差

第二段：
| 参数 | 含义 | 你的代码里的值 | 说明 |
|------|------|---------------|------|
| `root` | 数据集存放路径 | `"./data"` | 相对路径，表示在当前脚本同目录下创建一个 `data` 文件夹来存放数据 |
| `train` | 加载训练集还是测试集 | `True` | `True` = 训练集（5万张图），`False` = 测试集（1万张图） |
| `download` | 是否自动下载 | `True` | 首次运行会自动从官网下载，如果本地已有则跳过 |
| `transform` | 数据预处理方式 | `transform` | 就是你上一段代码里定义的那个流水线（ToTensor + Normalize） |

执行完这行代码后，train_dataset 就是一个数据集对象，你可以把它理解为一个"大列表

第四段：
1.整体功能——"自动配餐机器人"
DataLoader 就是一个数据搬运工。它不会改变数据本身，而是负责把 train_dataset 里的 5 万张图，按你设定的规则分批、打乱，然后一批一批地喂给模型。
打个比方：
train_dataset 是仓库，里面堆着 5 万张图
train_loader 是传送带，每次从仓库里抓一批图送到模型面前

2三个参数，只关注前两个
| 参数 | 含义 | 你的代码里的值 | 说明 |
|------|------|---------------|------|
| `train_dataset` | 数据来源 | 上一段代码创建的数据集对象 | 告诉 DataLoader 从哪拿数据 |
| `batch_size` | 每批多少张图 | 一个变量（比如 32、64、128） | 越大训练越快但越吃显存，越小训练越慢但更稳定 |
| `shuffle` | 是否打乱顺序 | `True` | 训练集必须打乱，防止模型"记住"样本顺序而不是学特征 |

# 3. Model
# =========================

model = resnet18(weights=None)

# CIFAR-10 images are only 32x32,
# so make the first layer smaller.
model.conv1 = nn.Conv2d(
    3,
    64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False
)

model.maxpool = nn.Identity()

# CIFAR-10 has 10 classes.
model.fc = nn.Linear(model.fc.in_features, 10)

model = model.to(device)

这段代码重点：
1.加载一个“空壳”ResNet18
model = resnet18(weights=None) 的意思是：从 torchvision 官方库里加载一个 ResNet18 模型，但不加载任何预训练权重，所有参数都是随机初始化的，相当于从零开始训练。

2.核心点二：改第一层卷积——适配小图
原始 ResNet18 是为 ImageNet 的 224×224 大图设计的，第一层卷积用的是 7×7 大卷积核 + stride=2，会把图片直接缩小一半。但 CIFAR-10 的图片只有 32×32，再用 7×7 就会丢失太多信息。所以改成 3×3 小卷积核 + stride=1 + padding=1，这样图片尺寸保持不变，尽可能保留原始信息。

参数含义：
3：输入通道数（RGB 三通道）
64：输出通道数（和原始模型保持一致）
kernel_size=3：卷积核大小从 7 改成 3
stride=1：步长从 2 改成 1，不再缩小图片
padding=1：填充 1 像素，保持尺寸不变
bias=False：后面接了 BatchNorm，不需要偏置

3.删掉最大池化层——防止特征图过小
model.maxpool = nn.Identity()
原始 ResNet18 在第一层卷积后面还有一个最大池化层，会把图片再缩小一半。对于 32×32 的小图来说，经过两次缩小后特征图就太小了，学不到有效特征。所以用 nn.Identity() 把它替换掉。
nn.Identity() 就是一个“占位层”，啥也不干，输入是什么输出就是什么，相当于把这个层“架空”了。

4.改最后一层全连接层——适配 10 分类
model.fc = nn.Linear(model.fc.in_features, 10)
原始 ResNet18 是为 ImageNet 的 1000 分类设计的，最后一层输出 1000 个节点。CIFAR-10 只有 10 个类别（飞机、汽车、猫、狗等），所以把输出改成 10。
model.fc.in_features：自动获取原始全连接层的输入特征数（512），不用自己数
10：输出 10 个类别的分数

📌 一句话总结
这段代码 = 拿一个空的 ResNet18 → 改第一层卷积适配 32×32 小图 → 删掉最大池化防止特征图过小 → 改最后一层输出 10 类 → 搬到 GPU 上准备训练。



# 4. Loss + Optimizer
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

这段代码详解：
1.criterion——“打分器”
riterion 就是交叉熵损失函数，作用是在每一轮训练中给模型的预测结果“打分”——预测得越准，分数（loss）越低；预测得越离谱，分数越高。

2.optimizer——“修正器”
optimizer 就是优化器，作用是根据 criterion 算出来的 loss，反向传播计算出梯度，然后按一定规则更新模型的所有参数，让下一轮预测更准。
两个关键参数：
model.parameters()：告诉优化器“要更新哪些参数”，这里传入的是整个 ResNet18 的所有可学习参数（卷积层的权重、全连接层的权重等）
lr（learning rate，学习率）：控制每次参数更新的“步长”。步长太大容易 overshoot（震荡不收敛），步长太小收敛太慢。Adam 的默认学习率通常是 0.001。
为什么选 Adam？因为它是自适应学习率优化器，结合了动量（Momentum）和 RMSprop 的优点，对超参数不敏感、收敛快，是大多数深度学习任务的默认首选。

#5. Training
1.双层循环结构——"外层轮次，内层批次"
for epoch in range(epochs):          # 外层循环：遍历轮次
    for images, labels in train_loader:  # 内层循环：遍历批次

外层 epoch（轮次）：表示把整个训练集完整看一遍。比如 epochs=10，模型就会把 5 万张图完整学习 10 轮。
内层 train_loader（批次）：每一轮里，数据被分成一个个小批次（batch），模型逐批学习。比如 batch_size=32，每轮就有约 1563 个批次。

2.核心点二：训练循环的"固定五步曲"
内层循环里的代码是 PyTorch 训练的标准模板，所有项目都是这个顺序，背下来就行：
# 第1步：数据搬到设备
images = images.to(device)
labels = labels.to(device)

# 第2步：前向传播 → 模型预测
outputs = model(images)

# 第3步：计算损失 → 打分器打分
loss = criterion(outputs, labels)

# 第4步：清空梯度 → 防止累加
optimizer.zero_grad()

# 第5步：反向传播 → 算梯度
loss.backward()

# 第6步：更新参数 → 修正器干活
optimizer.step()

3.model.train() 和损失打印
model.train()  # 切换到训练模式

print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")
轮训练结束后，打印平均损失：
total_loss：累加这一轮所有批次的 loss
len(train_loader)：批次总数
两者相除 = 这一轮的平均 loss

📌 一句话总结
这段代码 = 外层循环控制"学几轮"，内层循环按批次执行"前向→算损失→清梯度→反向传播→更新参数"五步曲，每轮结束后打印平均 loss 观察收敛情况。

# 6. Testing
1.model.eval() 和 torch.no_grad()——“考试模式”
这两行代码是测试时的固定搭配，相当于让模型进入“考试状态”：
model.eval()：切换到评估模式。让 Dropout 停止随机丢弃神经元（考试时不能随便丢信息），让 BatchNorm 使用训练好的统计量而不是当前批次的统计量。
torch.no_grad()：关闭梯度计算。测试时不需要反向传播更新参数，关掉梯度能节省大量显存和计算时间，速度会快很多。
⚠️ 注意：测试结束后如果想继续训练，必须调用 model.train() 切回训练模式，否则 Dropout 和 BatchNorm 会一直保持在“考试状态”。

2.torch.max(outputs, 1)——“从分数到类别”
_, predicted = torch.max(outputs, 1)
模型输出的 outputs 是原始分数（logits），比如 [2.1, -0.5, 3.7, ..., 1.2]，代表模型认为这张图属于每个类别的“得分”。这行代码的作用是找出得分最高的那个类别作为预测结果。
1：表示在第 1 个维度（类别维度）上找最大值
_：最大值本身（我们不需要，用下划线丢弃）
predicted：最大值对应的索引，也就是预测的类别（0~9）
打个比方：考试卷子上写了每道题的得分，torch.max 就是帮你找出得分最高的那道题，认为那就是正确答案。

3.准确率计算——“统计答对了几道
total += labels.size(0)
correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total

这三行是统计准确率的标准写法：
labels.size(0)：当前批次有多少张图（比如 batch_size=32）
predicted == labels：逐张图比较预测和真实标签，得到一个布尔数组（True/False）
.sum()：把 True 当成 1、False 当成 0，求和得到答对的数量
.item()：把 Tensor 里的标量值取出来变成 Python 数字
最后 correct / total * 100 就是准确率百分比

 一句话总结
这段代码 = 切换到评估模式 + 关闭梯度 → 逐批预测 → 找出得分最高的类别 → 统计答对的比例 → 打印准确率。